# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook guides you through exploring the FAIR² dataset using the `mlcroissant` library. You will:
- Load metadata & records from the dataset
- Review available record sets, fields, and schema structure by `@id`
- Load and process data with standard EDA steps
- Visualize and summarize findings

---

### Dataset Source
The dataset is defined via a Croissant schema JSON-LD file at:
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant is installed (uncomment if running in a new environment)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# The Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"(Version: {metadata.version})\nIdentifier: {metadata.identifier}\nDate Published: {metadata.datePublished}")
print(f"License: {metadata.license}")

## 2. Data Overview
List available record sets, their fields, and their `@id`s for referencing.

We'll enumerate the schema's record sets, showing for each the `@id`, display name, and their fields & columns `@id`s.

In [ ]:
# Find and display all record sets, fields, and columns by @id.
record_sets = dataset.record_sets

if len(record_sets) == 0:
    print("No record sets explicitly listed in the Croissant metadata.\n")
else:
    for rs in record_sets:
        print(f"Record set name: {rs.name}\n@id: {rs.id} (@type: {rs.type})")
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
        if rs.columns:
            print("  Columns:")
            for col in rs.columns:
                print(f"    - {col.name} (@id: {col.id}, type: {col.data_type})")
        print("")

# To illustrate, let's print a few example records from the first record set
if len(record_sets) > 0:
    example_record_set_id = record_sets[0].id
    print(f"\nExample records from record set: {example_record_set_id}\n---")
    for i, record in enumerate(dataset.records(record_set=example_record_set_id)):
        if i >= 3:
            break
        print(record)
else:
    print("No record sets defined; automatic loading is unavailable.")

## 3. Data Extraction
Load the entire data from each record set into `pandas.DataFrame`s.

We use the record set and field `@id`s found above for robust referencing.

In [ ]:
# Extract data for each available record set by @id
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if len(records) == 0:
        print(f"No records found for record set {rs_id}")
        continue
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records for record set {rs_id}")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(2))

# Pick the main record set for analysis
if record_set_ids:
    main_rs_id = record_set_ids[0]
    display_columns = list(dataframes[main_rs_id].columns)
    print(f"Available columns in primary record set ({main_rs_id}):\n", display_columns)
    dataframes[main_rs_id].head()
else:
    print("No record sets found to extract data from.")

## 4. Exploratory Data Analysis (EDA)
We demonstrate common data processing steps:
- Select and filter a numeric field by `@id`
- Normalize that field
- Group by a categorical field by `@id`

**All operations are referenced by the field or column `@id` as required by Croissant.**

In [ ]:
# Determine numeric and categorical field @id's from the loaded dataframe
df = dataframes[main_rs_id]
numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
print("Numeric columns candidates:", numeric_candidates)

# If the dataset is loaded with string representations, try parsing numbers
if not numeric_candidates:
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='ignore')
        except Exception:
            continue
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()

# Pick the first numeric field @id, or specify one by inspection
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
    print(f"Selected numeric field @id: {numeric_field_id}")
else:
    print("No numeric fields found to analyze.")
    numeric_field_id = None

# Choose a categorical/@id grouping field not the index; fallback to another string field
group_field_candidates = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
group_field = group_field_candidates[0] if group_field_candidates else None
if group_field:
    print(f"Selected group field @id: {group_field}")

if numeric_field_id:
    # Example: Threshold-based filtering
    # Set a threshold, e.g., filter those above the median value
    threshold = df[numeric_field_id].median()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field using z-score
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by the selected group field, if available
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].agg(['count','mean','std','min','max'])
        print(f"\nGrouped data by {group_field} (showing descriptive statistics):")
        print(grouped_df.head())
else:
    print("Skipping EDA: no suitable numeric fields found in primary record set.")

## 5. Visualization
Visualize the distribution of the numeric field, and relationship to the group-by field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field, y=numeric_field_id)
        plt.title(f"Distribution of {numeric_field_id} by {group_field}")
        plt.xticks(rotation=30, ha='right')
        plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion
In this notebook, we've:
- Loaded the dataset metadata and record structure using the `mlcroissant` library
- Explored record set, field, and column `@id`s for robust data referencing
- Loaded tabular dataset(s) and performed exploratory data analysis using only `@id`s for reference
- Filtered, normalized, grouped, and visualized the data, preparing it for downstream ML or clinical analysis

This approach demonstrates how Croissant enables reproducible, schema-driven FAIR data analysis workflows.